<a href="https://colab.research.google.com/github/sarahliu-mma/Quantitative-Trading-Pipeline_FIAM-Hackathon-2026/blob/main/notebooks/01_Build_stockpaneldata_universitypool_modeltable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build stock panel data, university pool, and model table








#### 1. Build stock panel data

In [36]:
import numpy as np
import pandas as pd

In [37]:
dt = pd.read_parquet("/content/drive/MyDrive/FIAM Hackathon/chars_final_with_names.parquet")
jkp_features = pd.read_csv("/content/drive/MyDrive/FIAM Hackathon/factor_char_list.csv")

In [38]:
dt.head()

,id,permno,permco,gvkey,iid,excntry,exch_main,common,primary_sec,bidask,...,qmj_prof,qmj_growth,qmj_safety,qmj,me_lag1,ticker,company_name,ticker_name_reference_date,ticker_name_source,ticker_name_status
0,10001,10001,7953,012994,01,USA,1,1,1,0,...,-0.793438,-1.174184,-0.554461,-1.339223,115.57776,EGAS,GAS NATURAL INC,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
1,10025,10025,7975,011903,01,USA,1,1,1,0,...,-0.659718,-1.648672,-1.076800,-1.539792,295.46015,AEPI,A E P INDUSTRIES INC,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
2,10026,10026,7976,012825,01,USA,1,1,1,0,...,1.291849,-1.083642,1.560033,1.060720,2031.60606,JJSF,J & J SNACK FOODS CORP,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
3,10028,10028,7978,012096,01,USA,1,1,1,0,...,0.288480,1.542084,-0.634534,0.691674,14.91328,DGSE,D G S E COMPANIES INC,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
4,10032,10032,7980,012945,01,USA,1,1,1,0,...,-0.891624,-0.555288,0.647570,-0.687090,1384.86205,PLXS,PLEXUS CORP,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month


In [39]:
IDENTIFIER = ['permno','eom']
FEATURES = jkp_features['variable'].tolist()
TARGET = ['ret_exc_lead1m']

In [40]:
all_col = dt.columns.tolist()
drop_col = ['id',
 'permco',
 'iid',
 'comp_exchg',
 'crsp_exchcd',
 'exchcd',
 'primaryexch',
 'crsp_shrcd',
 'shrcd',
 'comp_tpci',
 'conditionaltype',
 'excntry',
 'curcd',
 'fx',
 'ret_local',
 'ret',
 'ret_lag_dif',
 'prc_high',
 'prc_low',
 'adjfct',
 'me_company',
 'shares',
 'tvol',
 'sic',
 'naics',
 'gics',
 'source_crsp',
 'ticker_name_reference_date',
 'ticker_name_source',
 'ticker_name_status','ret_exc_lead1m']
reserved_col = [c for c in all_col if (c not in drop_col+FEATURES+IDENTIFIER)]

In [41]:
dt['eom'] = pd.to_datetime(dt['eom'])

In [42]:
dt.duplicated(['permno','eom']).sum()

np.int64(0)

In [43]:
def panel_data(dt, identifier, features, target):
  data = dt[identifier + features + target].copy()
  return data

In [44]:
panel_dt = panel_data(dt, IDENTIFIER, FEATURES+reserved_col, TARGET)

#### 2. Build university pool

In [45]:
panel_dt

,permno,eom,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,...,dolvol,ret_exc,obs_main,ff49,size_grp,ret_exc_wins,me_lag1,ticker,company_name,ret_exc_lead1m
0,10001,2015-01-31,349,0.527883,0.545502,0.024417,1.883066,0.059447,1.942361,0.683234,...,8.678426e+06,-0.115258,1,31.0,micro,-0.115258,1.155778e+02,EGAS,GAS NATURAL INC,0.035918
1,10025,2015-01-31,361,0.577454,0.448423,0.022546,5.879399,-0.052203,1.754731,2.597666,...,2.436483e+07,-0.138276,1,15.0,micro,-0.138276,2.954601e+02,AEPI,A E P INDUSTRIES INC,0.010997
2,10026,2015-01-31,361,0.571576,0.216947,0.001615,1.156135,0.066340,0.383311,1.363699,...,1.554623e+08,-0.097926,1,2.0,small,-0.097926,2.031606e+03,JJSF,J & J SNACK FOODS CORP,0.031309
3,10028,2015-01-31,361,0.522485,0.329818,9.495816,3.174725,-0.267115,0.985359,3.414208,...,1.553050e+05,0.237692,1,43.0,nano,0.237692,1.491328e+01,DGSE,D G S E COMPANIES INC,-0.072827
4,10032,2015-01-31,349,0.773526,0.548978,0.001747,1.959831,0.060762,1.229565,1.652661,...,1.637692e+08,-0.080576,1,37.0,small,-0.080576,1.384862e+03,PLXS,PLEXUS CORP,0.062307
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
529077,93372,2026-08-31,260,0.497253,NaN,0.001604,2.174204,0.080715,NaN,1.000764,...,2.674937e+08,-0.055245,1,21.0,small,-0.055245,1.020506e+03,PLOW,"DOUGLAS DYNAMICS, INC",NaN
529078,93374,2026-08-31,260,0.615893,NaN,0.000240,3.192214,0.163888,NaN,0.453439,...,8.890871e+08,-0.009834,1,46.0,large,-0.009834,7.656479e+03,None,None,NaN
529079,93397,2026-08-31,380,0.524713,NaN,0.014103,2.074070,-0.009369,NaN,0.448554,...,1.689150e+07,0.059552,1,1.0,micro,0.059552,2.351720e+02,LMNR,Limoneira CO,NaN
529080,93426,2026-08-31,260,0.566525,NaN,0.001584,1.339711,-0.031857,NaN,0.717224,...,9.672732e+08,-0.323378,1,37.0,micro,-0.323378,1.130190e+03,VPG,"Vishay Precision Group, Inc.",NaN


In [46]:
#Filter
def universe_filter(dt):
  c0 = dt['common'] == 1
  c1 = dt['primary_sec'] == 1
  c2 = dt['exch_main'] == 1
  c3 = (~dt['me'].isna())
  c4 = dt['prc']>= 5
  c5 = dt['bidask'] == 0
  c6 = dt['size_grp'] != 'nano'
  c7 = dt[FEATURES].isna().mean(axis = 1) <= 0.4
  universe_panel = dt.copy()[c0 & c1 & c2 & c3 & c4 & c5 & c6 & c7]

  return universe_panel

In [47]:
universe_panel = universe_filter(panel_dt)

In [48]:
universe_panel.to_parquet('/content/drive/MyDrive/FIAM Hackathon/universe_panel.parquet')

#### 3.Normalize the universe panel

In [49]:
universe_panel[FEATURES] = universe_panel.groupby('eom')[FEATURES].rank(pct = True)

In [50]:
universe_panel.head()

,permno,eom,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,...,dolvol,ret_exc,obs_main,ff49,size_grp,ret_exc_wins,me_lag1,ticker,company_name,ret_exc_lead1m
0,10001,2015-01-31,0.689607,0.319399,0.819856,0.814019,0.407899,0.483486,0.726124,0.515982,...,8.678426e+06,-0.115258,1,31.0,micro,-0.115258,115.57776,EGAS,GAS NATURAL INC,0.035918
1,10025,2015-01-31,0.712254,0.433847,0.703504,0.805565,0.802513,0.118412,0.697683,0.961012,...,2.436483e+07,-0.138276,1,15.0,micro,-0.138276,295.46015,AEPI,A E P INDUSTRIES INC,0.010997
2,10026,2015-01-31,0.712254,0.422890,0.276730,0.466009,0.059964,0.507027,0.176615,0.813839,...,1.554623e+08,-0.097926,1,2.0,small,-0.097926,2031.60606,JJSF,J & J SNACK FOODS CORP,0.031309
4,10032,2015-01-31,0.689607,0.755276,0.822552,0.480099,0.433034,0.490513,0.586025,0.873902,...,1.637692e+08,-0.080576,1,37.0,small,-0.080576,1384.86205,PLXS,PLEXUS CORP,0.062307
5,10044,2015-01-31,0.712254,0.808036,0.382300,0.901726,0.374506,0.906535,0.224017,0.806814,...,5.329766e+06,0.038859,1,2.0,micro,0.038859,80.07136,RMCF,ROCKY MOUNTAIN CHOCOLATE FAC INC,0.129391


In [51]:
model_table = universe_panel[IDENTIFIER + FEATURES + TARGET]

In [52]:
model_table.to_parquet('/content/drive/MyDrive/FIAM Hackathon/model_table.parquet')